# Lab 4 · EDA dan Leakage Check pada PathMNIST

Terkait: **Bab 04 · Validasi Data dan Pra-pemrosesan**

## Tujuan
1. Download PathMNIST (histologi 9 kelas) dan lakukan EDA 3 lapis.
2. Audit 5 jenis data leakage.
3. Verifikasi bahwa preprocessing fit hanya pada train set.
4. Tulis audit report yang jujur, bukan centang ritual.

## Prasyarat
```bash
pip install -e '.[medical]'
```

## Checklist
- [ ] EDA 3 lapis: shape/statistik, distribusi kelas, relasi warna-kelas.
- [ ] Audit train-test overlap (hash-based).
- [ ] Verifikasi mean/std dihitung dari train only.
- [ ] Audit report ditulis sebagai paragraf, bukan centang.

## 0. Setup dan muat data

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.data import build_loaders

cfg = {
    'name': 'pathmnist',
    'root': '../data',
    'image_size': 28,
    'batch_size': 256,
    'num_workers': 0,
    'val_split': 0.0,  # pakai split resmi PathMNIST
    'augment': False,
}
train_loader, val_loader, test_loader = build_loaders(cfg)
print(f'train: {len(train_loader.dataset)} | val: {len(val_loader.dataset)} | test: {len(test_loader.dataset)}')

## 1. EDA Lapis 1: Shape dan Statistik Dasar

In [ ]:
# Ambil 1 batch dan periksa properti dasar
x_batch, y_batch = next(iter(train_loader))
y_batch = torch.as_tensor(y_batch).long().view(-1)

print('=== Shape dan Tipe ===')
print(f'Input shape (batch): {x_batch.shape}')
print(f'Label shape (batch): {y_batch.shape}')
print(f'Dtype: {x_batch.dtype}')
print(f'Label dtype: {y_batch.dtype}')

print('\n=== Statistik Pixel (batch pertama) ===')
print(f'Min:  {x_batch.min():.4f}')
print(f'Max:  {x_batch.max():.4f}')
print(f'Mean (per channel): {x_batch.mean(dim=[0,2,3]).numpy().round(4)}')
print(f'Std  (per channel): {x_batch.std(dim=[0,2,3]).numpy().round(4)}')

print('\n=== Label unik ===')
print(f'Label values: {torch.unique(y_batch).numpy()}')

In [ ]:
# Hitung statistik pixel dari seluruh train set (penting untuk normalisasi)
# Ini yang harus dipakai sebagai mean/std normalisasi — BUKAN dari val/test

channel_sum = torch.zeros(3)
channel_sq_sum = torch.zeros(3)
n_pixels = 0

for x, _ in train_loader:
    b, c, h, w = x.shape
    channel_sum += x.mean(dim=[0, 2, 3]) * b
    channel_sq_sum += (x ** 2).mean(dim=[0, 2, 3]) * b
    n_pixels += b

mean_train = channel_sum / n_pixels
std_train = torch.sqrt(channel_sq_sum / n_pixels - mean_train ** 2)

print('=== Mean/Std dari TRAIN set (untuk normalisasi) ===')
print(f'mean: {mean_train.numpy().round(4)}')
print(f'std:  {std_train.numpy().round(4)}')
print('\n⚠ Pastikan nilai ini yang dipakai di src/data.py untuk PathMNIST,')
print('  bukan hardcode dari dataset lain!')

## 2. EDA Lapis 2: Distribusi Kelas

In [ ]:
# Hitung distribusi kelas di train, val, test
from collections import Counter

def count_classes(loader, n_classes=9):
    counts = Counter()
    for _, y in loader:
        y = torch.as_tensor(y).long().view(-1)
        counts.update(y.numpy().tolist())
    return [counts.get(i, 0) for i in range(n_classes)]

train_counts = count_classes(train_loader)
val_counts   = count_classes(val_loader)
test_counts  = count_classes(test_loader)

PATHMNIST_CLASSES = [
    'adipose', 'background', 'debris', 'lymphocytes',
    'mucus', 'smooth muscle', 'normal colon', 'cancer stroma', 'colorectal adenocarcinoma'
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, counts, title in zip(axes, [train_counts, val_counts, test_counts], ['Train', 'Val', 'Test']):
    bars = ax.bar(range(9), counts, color='steelblue', alpha=0.8)
    ax.set_xticks(range(9))
    ax.set_xticklabels([c[:8] for c in PATHMNIST_CLASSES], rotation=45, ha='right', fontsize=8)
    ax.set_title(f'{title} (n={sum(counts)})')
    ax.set_ylabel('Jumlah sampel')
    # Annotate jumlah
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                str(count), ha='center', va='bottom', fontsize=7)

plt.suptitle('Distribusi Kelas PathMNIST', fontsize=11)
plt.tight_layout()
plt.show()

# Laporan imbalance
total_train = sum(train_counts)
print('\nProposi kelas di train set:')
for i, (cls, cnt) in enumerate(zip(PATHMNIST_CLASSES, train_counts)):
    pct = cnt / total_train * 100
    flag = '⚠' if pct < 5 else ' '
    print(f'  {flag} {i}: {cls:30s} {cnt:6d} ({pct:5.1f}%)')

## 3. EDA Lapis 3: Relasi Warna dan Kelas

In [ ]:
# Visualisasi sampel per kelas
class_samples = {i: [] for i in range(9)}
for x, y in train_loader:
    y = torch.as_tensor(y).long().view(-1)
    for img, label in zip(x, y):
        label = label.item()
        if len(class_samples[label]) < 3:
            class_samples[label].append(img)
    if all(len(v) >= 3 for v in class_samples.values()):
        break

# Denormalisasi
mean_t = torch.tensor(mean_train).view(3, 1, 1)
std_t  = torch.tensor(std_train).view(3, 1, 1)

fig, axes = plt.subplots(9, 3, figsize=(6, 18))
for row, (cls_id, imgs) in enumerate(class_samples.items()):
    for col, img in enumerate(imgs[:3]):
        denorm = (img * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[row][col].imshow(denorm)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(PATHMNIST_CLASSES[cls_id][:12], rotation=0,
                                       labelpad=70, fontsize=8, va='center')
plt.suptitle('3 Contoh per Kelas PathMNIST', fontsize=11)
plt.tight_layout()
plt.savefig('../experiments/pathmnist_samples.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Apakah kelas bisa dipisahkan hanya dari intensitas rata-rata?
# Hitung mean intensitas per channel per kelas

class_intensities = {i: [] for i in range(9)}
for x, y in train_loader:
    y = torch.as_tensor(y).long().view(-1)
    for img, label in zip(x, y):
        mean_rgb = img.mean(dim=[1, 2]).numpy()  # shape (3,)
        class_intensities[label.item()].append(mean_rgb)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
channel_names = ['Red', 'Green', 'Blue']
for ch_idx, (ax, ch_name) in enumerate(zip(axes, channel_names)):
    data_per_class = [np.array([v[ch_idx] for v in class_intensities[i]]) for i in range(9)]
    ax.boxplot(data_per_class, labels=[f'{i}' for i in range(9)])
    ax.set_xlabel('Kelas')
    ax.set_ylabel('Mean intensity')
    ax.set_title(f'Channel {ch_name}')
    ax.grid(alpha=0.3)

plt.suptitle('Distribusi Intensitas per Kelas PathMNIST', fontsize=11)
plt.tight_layout()
plt.show()

print('Pertanyaan: Kelas mana yang paling berbeda secara warna? Apakah perbedaan itu cukup untuk klasifikasi sederhana?')

## 4. Audit Leakage: Train-Test Overlap

In [ ]:
import hashlib

def tensor_hash(t: torch.Tensor) -> str:
    """Hash tensor bytes — cepat untuk mendeteksi duplikasi exact."""
    return hashlib.md5(t.numpy().tobytes()).hexdigest()

print('Menghitung hash train split...')
train_hashes = set()
for x, _ in train_loader:
    for img in x:
        train_hashes.add(tensor_hash(img))

print('Menghitung hash test split...')
overlap_count = 0
total_test = 0
for x, _ in test_loader:
    for img in x:
        total_test += 1
        if tensor_hash(img) in train_hashes:
            overlap_count += 1

print(f'\nTotal test samples: {total_test}')
print(f'Train-test overlap (exact): {overlap_count}')
if overlap_count == 0:
    print('✓ Tidak ada duplikasi exact antara train dan test.')
else:
    print(f'⚠ Ada {overlap_count} sampel test yang identik dengan train! Ini data leakage.')

## 5. Audit Leakage: Preprocessing Pipeline

In [ ]:
# Verifikasi bahwa normalisasi yang dipakai di data.py menggunakan
# statistik dari train set, bukan dari seluruh dataset atau hardcode dari CIFAR-10

import inspect
from src import data as data_module

source = inspect.getsource(data_module)

# Cari apakah ada referensi ke PathMNIST-specific normalization
print('=== Cek src/data.py untuk normalisasi PathMNIST ===')
lines_with_normalize = [l.strip() for l in source.split('\n') if 'Normalize' in l or 'pathmnist' in l.lower()]
for line in lines_with_normalize:
    print(' ', line)

print()
print('Statistik dari train set yang dihitung di atas:')
print(f'  mean: {mean_train.numpy().round(4)}')
print(f'  std:  {std_train.numpy().round(4)}')
print()
print('Catatan: Jika src/data.py menggunakan nilai CIFAR-10 untuk PathMNIST,')
print('itu adalah preprocessing leakage (menggunakan asumsi dataset berbeda).')

## 6. Audit Leakage: Temporal dan Group (Patient)

Dua jenis leakage ini relevan untuk data medis:

### 6a. Temporal Leakage

PathMNIST tidak menyertakan timestamp gambar dalam metadata publik. Oleh karena itu:

- Kita tidak dapat memverifikasi apakah gambar di test set diambil setelah train set.
- Ini adalah **keterbatasan yang harus disebutkan di laporan**, bukan disembunyikan.
- Implikasi: jika model di-deploy ke data real, distribusi histologi bisa berubah seiring waktu (concept drift).

### 6b. Group Leakage (Patient-Level)

Idealnya, semua patch dari satu pasien harus masuk ke split yang sama (train ATAU test, tidak keduanya). Jika satu pasien punya patch di train dan test, model bisa belajar pasien-spesifik, bukan histologi-generik.

In [ ]:
# Cek apakah MedMNIST menyediakan patient metadata
try:
    import medmnist
    from medmnist import PathMNIST
    ds = PathMNIST(split='train', download=False, root='../data')
    # Cek atribut yang tersedia
    attrs = [a for a in dir(ds) if not a.startswith('_')]
    print('Atribut PathMNIST dataset:', attrs)
    if hasattr(ds, 'info'):
        print('\nInfo:', ds.info)
except Exception as e:
    print(f'Tidak bisa mengakses medmnist langsung: {e}')

print()
print('Catatan: MedMNIST v2 tidak menyertakan patient ID dalam dataset publik.')
print('Ini adalah keterbatasan yang perlu disebutkan di audit report.')

## 7. Audit Report

Tulis sebagai paragraf jujur di bawah. Bukan centang ritual — tulis apa yang kamu temukan dan apa artinya.

### Audit Report PathMNIST

> *[Contoh format — ganti dengan temuanmu sendiri]*
>
> **Ringkasan dataset:** PathMNIST berisi X sampel train, Y sampel test, 9 kelas histologi kolorektal. Distribusi kelas tidak seimbang: kelas terbesar (adipose) menempati ~X% sampel, kelas terkecil (debris) hanya ~X%.
>
> **Audit overlap:** Tidak ditemukan duplikasi exact (hash MD5) antara train dan test split. Kemungkinan near-duplicate tidak diperiksa dalam lab ini karena membutuhkan embedding similarity — ini keterbatasan audit.
>
> **Audit preprocessing:** Normalisasi menggunakan mean/std yang dihitung dari train set saja (diverifikasi dari src/data.py). Tidak ada preprocessing leakage yang terdeteksi.
>
> **Audit temporal:** PathMNIST tidak menyediakan timestamp — temporal leakage tidak dapat diverifikasi. Ini keterbatasan yang perlu disebutkan di laporan eksperimen.
>
> **Audit group (patient):** Patient ID tidak tersedia di dataset publik. Split dilakukan di level patch, bukan patient. Artinya model bisa belajar fitur per-pasien, dan generalisasi ke pasien baru mungkin lebih rendah dari yang dilaporkan metrik ini.
>
> **Kesimpulan untuk eksperimen:** Dataset aman dipakai dengan catatan: (1) metrik harus diinterpretasi sebagai patch-level accuracy, bukan patient-level; (2) imbalance kelas perlu ditangani dengan stratified sampling atau weighted loss.

## 8. Refleksi

1. Dari tiga EDA lapis yang dilakukan, lapisan mana yang paling mengejutkan? Apa satu fakta tentang data yang tidak kamu duga sebelum melihatnya?

2. Audit patient-level leakage tidak bisa dilakukan karena data tidak tersedia. Bayangkan kamu adalah reviewer paper yang melihat hasil akurasi tinggi pada PathMNIST tanpa audit ini — apa satu pertanyaan yang akan kamu ajukan?

3. Jika kamu mengetahui ada 5% label yang mungkin salah di train set, apa yang akan kamu lakukan: (a) abaikan karena 5% kecil, (b) bersihkan semua, atau (c) eksperimen dengan dan tanpa pembersihan? Jelaskan alasanmu.

### Jawaban Refleksi

**1. Temuan paling mengejutkan:**
> *[tulis di sini]*

**2. Pertanyaan untuk paper tanpa patient audit:**
> *[tulis di sini]*

**3. Strategi menghadapi 5% label noise:**
> *[tulis di sini]*